In [1]:
# import current working directory

import os

print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\research\model_experiments


In [2]:
# import libraries

import pandas as pd
import numpy as np

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss
)

In [3]:
# scaled transformed data will be used for svm

train_df = pd.read_csv(
    "../../artifacts/data_transformation/train.csv"
)

test_df = pd.read_csv(
    "../../artifacts/data_transformation/test.csv"
)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (95512, 917)
Test shape : (23878, 917)


In [4]:
# seperate features and target 

X_train = train_df.drop("is_canceled", axis=1)

y_train = train_df["is_canceled"]

X_test = test_df.drop( "is_canceled",axis=1)

y_test = test_df["is_canceled"]

In [5]:
# checking shapes 

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape :", X_test.shape)
print("y_test shape :", y_test.shape)

X_train shape: (95512, 916)
y_train shape: (95512,)
X_test shape : (23878, 916)
y_test shape : (23878,)


In [6]:
# checking target distribution

print("Training target distribution:")
print(y_train.value_counts())

print("\nTraining target percentage:")
print(y_train.value_counts(normalize=True) * 100)

Training target distribution:
is_canceled
0    60133
1    35379
Name: count, dtype: int64

Training target percentage:
is_canceled
0    62.958581
1    37.041419
Name: proportion, dtype: float64


# SVM Baseline

In [7]:
svm_base = LinearSVC(   # it is specifically designed for large-scale linear SVM problems
    C=1.0,   # This is our baseline/default regularization value
    random_state=42,  
    max_iter=5000
)

In [8]:
# train svm 

svm_base.fit(X_train,y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


In [9]:
# make predictions

svm_pred = svm_base.predict(X_test)

svm_score = svm_base.decision_function(X_test)

In [10]:
# checking probability calibatation for class 

# The Linear SVM gives decision score.Calibration converts  scores into probability estimates

svm_calibrated = CalibratedClassifierCV(
    svm_base,
    cv=3,
    method="sigmoid",
    n_jobs=-1
)

In [11]:
# fitting calibration

svm_calibrated.fit( X_train,y_train)

,estimator,LinearSVC(max...ndom_state=42)
,method,'sigmoid'
,cv,3
,n_jobs,-1
,ensemble,'auto'
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'


In [12]:
# calibrated predictions

svm_pred = svm_calibrated.predict(X_test)

svm_prob = svm_calibrated.predict_proba( X_test)[:, 1]

In [13]:
# evaluation metrix

# Confusion Matrix
svm_cm = confusion_matrix(
    y_test,
    svm_pred
)

tn, fp, fn, tp = svm_cm.ravel()


# 1. Accuracy
svm_accuracy = accuracy_score(
    y_test,
    svm_pred
)


# 2. Precision
svm_precision = precision_score(
    y_test,
    svm_pred,
    zero_division=0
)


# 3. Recall
svm_recall = recall_score(
    y_test,
    svm_pred,
    zero_division=0
)


# 4. Specificity
svm_specificity = (
    tn / (tn + fp)
)


# 5. F1 Score
svm_f1 = f1_score(
    y_test,
    svm_pred,
    zero_division=0
)

# 6. ROC-AUC
svm_roc_auc = roc_auc_score(
    y_test,
    svm_prob
)

# 7. PR-AUC
svm_pr_auc = average_precision_score(
    y_test,
    svm_prob
)

# 8. Log Loss

svm_logloss = log_loss(
    y_test,
    svm_prob
)


In [14]:
# Print all metrics

print("=" * 55)
print("LINEAR SVM - BASELINE EVALUATION")
print("=" * 55)

print(f"Accuracy    : {svm_accuracy:.4f}")
print(f"Precision   : {svm_precision:.4f}")
print(f"Recall      : {svm_recall:.4f}")
print(f"Specificity : {svm_specificity:.4f}")
print(f"F1 Score    : {svm_f1:.4f}")
print(f"ROC-AUC     : {svm_roc_auc:.4f}")
print(f"PR-AUC      : {svm_pr_auc:.4f}")
print(f"Log Loss    : {svm_logloss:.4f}")

LINEAR SVM - BASELINE EVALUATION
Accuracy    : 0.8322
Precision   : 0.8118
Recall      : 0.7122
Specificity : 0.9029
F1 Score    : 0.7587
ROC-AUC     : 0.9113
PR-AUC      : 0.8711
Log Loss    : 0.3574


In [15]:
# save baseline model

svm_baseline_results = pd.DataFrame({
    "Model": ["Linear SVM"],
    "Accuracy": [svm_accuracy],
    "Precision": [svm_precision],
    "Recall": [svm_recall],
    "Specificity": [svm_specificity],
    "F1 Score": [svm_f1],
    "ROC-AUC": [svm_roc_auc],
    "PR-AUC": [svm_pr_auc],
    "Log Loss": [svm_logloss]
})

svm_baseline_results.round(4)

,Model,Accuracy,Precision,Recall,Specificity,F1 Score,ROC-AUC,PR-AUC,Log Loss
0,Linear SVM,0.8322,0.8118,0.7122,0.9029,0.7587,0.9113,0.8711,0.3574


# hyper parameter tuning

In [16]:
# c experiment

C_values = [0.01, 0.1, 1, 10, 100]

In [17]:
C_results = []

for c_value in C_values:
    print("Training C =", c_value)

model = LinearSVC(
        C=c_value,
        random_state=42,
        max_iter=5000)

Training C = 0.01
Training C = 0.1
Training C = 1
Training C = 10
Training C = 100


In [18]:
# fit linearsvc model
model.fit(X_train,y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,100
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


In [19]:
# evalutaion metrix

# C experiment

C_values = [0.01, 0.1, 1, 10, 100]

C_results = []

for c_value in C_values:

    print("Training C =", c_value)

    model = LinearSVC(
        C=c_value,
        random_state=42,
        max_iter=5000
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_test
    )

    score = model.decision_function(
        X_test
    )

    # Confusion Matrix
    cm = confusion_matrix(
        y_test,
        pred
    )

    tn, fp, fn, tp = cm.ravel()

    # Metrics
    accuracy_value = accuracy_score(
        y_test,
        pred
    )

    precision_value = precision_score(
        y_test,
        pred,
        zero_division=0
    )

    recall_value = recall_score(
        y_test,
        pred,
        zero_division=0
    )

    specificity_value = (
        tn / (tn + fp)
    )

    f1_value = f1_score(
        y_test,
        pred,
        zero_division=0
    )

    roc_auc_value = roc_auc_score(
        y_test,
        score
    )

    pr_auc_value = average_precision_score(
        y_test,
        score
    )

    C_results.append({
        "C": c_value,
        "Accuracy": accuracy_value,
        "Precision": precision_value,
        "Recall": recall_value,
        "Specificity": specificity_value,
        "F1 Score": f1_value,
        "ROC-AUC": roc_auc_value,
        "PR-AUC": pr_auc_value
    })

print("\nC experiment completed.")

   

Training C = 0.01
Training C = 0.1
Training C = 1
Training C = 10
Training C = 100

C experiment completed.


In [20]:
# Print all metrics

print("=" * 55)
print("LINEAR SVM - C EVALUATION")
print("=" * 55)

print(f"Accuracy    : {svm_accuracy:.4f}")
print(f"Precision   : {svm_precision:.4f}")
print(f"Recall      : {svm_recall:.4f}")
print(f"Specificity : {svm_specificity:.4f}")
print(f"F1 Score    : {svm_f1:.4f}")
print(f"ROC-AUC     : {svm_roc_auc:.4f}")
print(f"PR-AUC      : {svm_pr_auc:.4f}")
print(f"Log Loss    : {svm_logloss:.4f}")

LINEAR SVM - C EVALUATION
Accuracy    : 0.8322
Precision   : 0.8118
Recall      : 0.7122
Specificity : 0.9029
F1 Score    : 0.7587
ROC-AUC     : 0.9113
PR-AUC      : 0.8711
Log Loss    : 0.3574


In [21]:
# display results 

C_results_df = pd.DataFrame(C_results)

C_results_df.round(4)

,C,Accuracy,Precision,Recall,Specificity,F1 Score,ROC-AUC,PR-AUC
0,0.01,0.8304,0.8209,0.6934,0.9110,0.7518,0.9100,0.8688
1,0.10,0.8324,0.8171,0.7056,0.9071,0.7573,0.9114,0.8712
2,1.00,0.8316,0.8139,0.7070,0.9049,0.7567,0.9112,0.8710
3,10.00,0.8316,0.8131,0.7082,0.9042,0.7570,0.9110,0.8708
4,100.00,0.8316,0.8129,0.7086,0.9040,0.7572,0.9109,0.8708


In [22]:
# import gridsearch cv 

from sklearn.model_selection import GridSearchCV

In [23]:
# parameter grid

svm_param_grid = {
    "C": [0.1],
    "class_weight": [None, "balanced"]
}

In [24]:
svm_grid_search = GridSearchCV(
    estimator=LinearSVC(
        random_state=42,
        max_iter=5000
    ),
    param_grid=svm_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

In [25]:
svm_grid_search.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 2 candidates, totalling 10 fits


,estimator,LinearSVC(max...ndom_state=42)
,param_grid,"{'C': [0.1], 'class_weight': [None, 'balanced']}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [26]:
# parameter

print("Best Parameters:")
print(svm_grid_search.best_params_)

print("\nBest CV F1 Score:")
print(svm_grid_search.best_score_)

Best Parameters:
{'C': 0.1, 'class_weight': 'balanced'}

Best CV F1 Score:
0.7787921355669056


In [27]:
# tuned model

best_svm_model = svm_grid_search.best_estimator_

In [28]:
# prediction and tuned score

svm_tuned_pred = best_svm_model.predict(X_test)

svm_tuned_score = best_svm_model.decision_function(X_test)

In [29]:
# calibarating tuned svm

svm_tuned_calibrated = CalibratedClassifierCV(
    best_svm_model,
    cv=3,
    method="sigmoid",
    n_jobs=-1
)

svm_tuned_calibrated.fit(
    X_train,
    y_train
)


,estimator,LinearSVC(C=0...ndom_state=42)
,method,'sigmoid'
,cv,3
,n_jobs,-1
,ensemble,'auto'
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,0.1
,multi_class,'ovr'


In [30]:
# prediction and probability

svm_tuned_pred = svm_tuned_calibrated.predict( X_test)

svm_tuned_prob = svm_tuned_calibrated.predict_proba(X_test)[:, 1]


In [31]:
# evaluation metrix

svm_tuned_cm = confusion_matrix(
    y_test,
    svm_tuned_pred
)

tn, fp, fn, tp = svm_tuned_cm.ravel()


svm_tuned_accuracy = accuracy_score(
    y_test,
    svm_tuned_pred
)

svm_tuned_precision = precision_score(
    y_test,
    svm_tuned_pred,
    zero_division=0
)

svm_tuned_recall = recall_score(
    y_test,
    svm_tuned_pred,
    zero_division=0
)

svm_tuned_specificity = tn / (tn + fp)

svm_tuned_f1 = f1_score(
    y_test,
    svm_tuned_pred,
    zero_division=0
)

svm_tuned_roc_auc = roc_auc_score(
    y_test,
    svm_tuned_prob
)

svm_tuned_pr_auc = average_precision_score(
    y_test,
    svm_tuned_prob
)

svm_tuned_logloss = log_loss(
    y_test,
    svm_tuned_prob
)


In [32]:
# display results 

print("=" * 55)
print("LINEAR SVM - TUNED MODEL EVALUATION")
print("=" * 55)

print(f"Accuracy    : {svm_tuned_accuracy:.4f}")
print(f"Precision   : {svm_tuned_precision:.4f}")
print(f"Recall      : {svm_tuned_recall:.4f}")
print(f"Specificity : {svm_tuned_specificity:.4f}")
print(f"F1 Score    : {svm_tuned_f1:.4f}")
print(f"ROC-AUC     : {svm_tuned_roc_auc:.4f}")
print(f"PR-AUC      : {svm_tuned_pr_auc:.4f}")
print(f"Log Loss    : {svm_tuned_logloss:.4f}")

LINEAR SVM - TUNED MODEL EVALUATION
Accuracy    : 0.8316
Precision   : 0.8071
Recall      : 0.7167
Specificity : 0.8992
F1 Score    : 0.7592
ROC-AUC     : 0.9120
PR-AUC      : 0.8689
Log Loss    : 0.3579


In [33]:
# baseline vs tuned model comparing

svm_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1 Score",
        "ROC-AUC",
        "PR-AUC",
        "Log Loss"
    ],
    
    "Baseline": [
        svm_accuracy,
        svm_precision,
        svm_recall,
        svm_specificity,
        svm_f1,
        svm_roc_auc,
        svm_pr_auc,
        svm_logloss
    ],
    
    "Tuned": [
        svm_tuned_accuracy,
        svm_tuned_precision,
        svm_tuned_recall,
        svm_tuned_specificity,
        svm_tuned_f1,
        svm_tuned_roc_auc,
        svm_tuned_pr_auc,
        svm_tuned_logloss
    ]
})

svm_comparison.round(4)

,Metric,Baseline,Tuned
0,Accuracy,0.8322,0.8316
1,Precision,0.8118,0.8071
2,Recall,0.7122,0.7167
3,Specificity,0.9029,0.8992
4,F1 Score,0.7587,0.7592
5,ROC-AUC,0.9113,0.9120
6,PR-AUC,0.8711,0.8689
7,Log Loss,0.3574,0.3579
